# 07 — Cash-flow projector (Tier 3, second estimator)

Venture-specific, population-blind: fits each company's OWN recent burn
trend and projects forward to a zero-cash-balance date. Deliberately the
opposite of the survival estimator (which is population-informed) --
the IDF's Tier 4 fusion is what reconciles the two when they disagree.

Reads: `data/processed/synthetic_trajectories.csv`, `data/processed/outcomes.csv`
Writes: `data/processed/cashflow_projections.csv`

In [1]:
import pandas as pd
import numpy as np

PROCESSED = "../data/processed"

traj = pd.read_csv(f"{PROCESSED}/synthetic_trajectories.csv")
outcomes = pd.read_csv(f"{PROCESSED}/outcomes.csv")
print(f"[load] {traj['object_id'].nunique():,} companies with synthesized trajectories")

[load] 1,896 companies with synthesized trajectories


In [2]:
def project_zero_crossing(spend_series, starting_cash, trend_window=6):
    """Fit a simple linear trend to the trailing `trend_window` months of
    spend, then project the balance forward until it crosses zero."""
    recent = spend_series[-trend_window:] if len(spend_series) >= trend_window else spend_series
    months = np.arange(len(recent))
    if len(recent) >= 2:
        slope, intercept = np.polyfit(months, recent, 1)
    else:
        slope, intercept = 0.0, recent.mean() if len(recent) else 0.0

    balance = starting_cash
    m = len(spend_series)
    projected_spend = intercept + slope * (len(recent) - 1)
    while balance > 0 and m < len(spend_series) + 600:  # 50-year safety cap
        projected_spend = max(projected_spend + slope, 1.0)
        balance -= projected_spend
        m += 1
    months_to_zero = m - len(spend_series)
    return months_to_zero, slope

results = []
for obj_id, g in traj.groupby("object_id"):
    g = g.sort_values("month_idx")
    spend = g["synthesized_spend"].values
    starting_cash = spend[-1] * 3  # placeholder: assume ~3 months of runway remained at last observation
    months_to_zero, trend = project_zero_crossing(spend, starting_cash)
    results.append({"object_id": obj_id, "months_to_zero_projected": months_to_zero, "burn_trend": trend})

projections = pd.DataFrame(results)
projections.to_csv(f"{PROCESSED}/cashflow_projections.csv", index=False)
print(f"[done] projected zero-crossing for {len(projections):,} companies")
print(f"[done] median months-to-zero (projected): {projections['months_to_zero_projected'].median():.1f}")
print(f"[done] wrote {PROCESSED}/cashflow_projections.csv")
print()
print("NOTE: 'starting_cash' above is a placeholder (3x last month's spend) because")
print("actual current cash balance is only available from founder-supplied intake,")
print("not from the Crunchbase snapshot. Replace with real intake data once the")
print("founder-facing form exists -- this notebook's projection LOGIC is real and")
print("tested; only this one input is a stand-in.")

[done] projected zero-crossing for 1,896 companies
[done] median months-to-zero (projected): 4.0
[done] wrote ../data/processed/cashflow_projections.csv

NOTE: 'starting_cash' above is a placeholder (3x last month's spend) because
actual current cash balance is only available from founder-supplied intake,
not from the Crunchbase snapshot. Replace with real intake data once the
founder-facing form exists -- this notebook's projection LOGIC is real and
tested; only this one input is a stand-in.
